# 🌍 Global Tech Salary & Workforce Trends Analytics

## 📖 Background & Objective
In the highly competitive tech industry, attracting and retaining top talent requires a data-driven approach to compensation and workforce planning. Working as a data consultant for an international HR firm, this project analyzes a comprehensive dataset of **over 57,000 global tech salary records** spanning from 2020 to 2024.

The goal is to uncover macro-level insights into global compensation drivers, the evolving reality of remote work, and international hiring strategies across more than 200 unique tech roles to guide strategic decision-making.

## 🛠️ Tech Stack & Methodology
* **Language/Engine:** SQL (PostgreSQL syntax via DuckDB)
* **Environment:** Kaggle Notebook integrated with Python (Pandas)
* **Advanced SQL Techniques used:** Common Table Expressions (CTEs), Window Functions, Conditional Aggregations (`CASE WHEN`), and Multi-level Joins.

---

## Executive Summary & Table of Contents

### Environment Setup

### 1. Theme 1: Job Market Dynamics & Evolution

- 1.1 Market Giants & Trend Shifts

- 1.2 The High-Earning Elite

- 1.3 Emerging Tech Roles

### 2. Theme 2: Company Size vs. Compensation

- 2.1 The Corporate Salary Premium

- 2.2 Hiring Preferences (Startups vs. Corporates)

### 3. Theme 3: The Post-Pandemic Remote Work Paradox

- 3.1 Remote vs. In-Person Salary Comparison

- 3.2 Remote Flexibility by Company Size

### 4. Theme 4: Career Progression ROI

- 4.1 The Seniority Salary Leap

### 5. Conclusions & Strategic Recommendations

- 5.1 The Talent & Compensation Landscape

- 5.2 Corporate Architecture vs. Startups

- 5.3 The Reality of Modern Work Models

### Author's Contact & Thank You Note

---

# 🛠️ Environment Setup
In this section, we load the dataset and configure **DuckDB** to run standard **SQL** queries directly over our data frames.

In [1]:
import duckdb
import kagglehub
import pandas as pd

con = duckdb.connect(database=':memory:')

# downloading the dataset
path_new = kagglehub.dataset_download("adillalopes/salaries")

# loads the dataset as a dataframe
def get_data():
  data = pd.read_csv(f'{path_new}/salaries.csv', encoding='latin-1')
  return data

# runs SQL query with duckdb
def run_sql(query):
    return duckdb.query(query).df()

# loading data into salaries
salaries = get_data()

---

# 📈 1. Theme 1: Job Market Dynamics & Evolution

## 📊 Q1.1: Market Giants & Trend Shifts
What are the top 5 job titles with the highest concentration of professionals in the current global tech market? Has their dominance shifted between 2020 and 2024?

In [2]:
run_sql("""
        WITH job_count_per_year AS (
            SELECT 
                work_year,
                job_title,
                COUNT(*) AS total_employees,
-- creates a rank from 1 to N ordered from the most popular job_titles per year
                DENSE_RANK() OVER (
                    PARTITION BY work_year 
                    ORDER BY COUNT(*) DESC) AS year_rank
                FROM salaries
                GROUP BY work_year, job_title
            )
        SELECT 
            work_year,
            year_rank,
            job_title,
            total_employees
        FROM job_count_per_year
        -- filtering to get only the top 5 from every year
        WHERE year_rank <= 5
        ORDER BY work_year ASC, year_Rank ASC;
""")

,work_year,year_rank,job_title,total_employees
0,2020,1,Data Scientist,25
1,2020,2,Data Engineer,13
2,2020,3,Data Analyst,6
3,2020,4,Machine Learning Engineer,5
4,2020,5,Business Data Analyst,3
5,2020,5,Big Data Engineer,3
6,2021,1,Data Scientist,61
7,2021,2,Data Engineer,37
8,2021,3,Machine Learning Engineer,22
9,2021,4,Data Analyst,20


💡 Data Insight:

- **Data-Centric Dominance:** Data-focused roles (such as *Data Scientist* and *Data Engineer*) have consistently dominated the top 3 positions since 2020, underscoring the sustained global demand for data infrastructure and analytics.

- **The Rise of AI/ML:** **Machine Learning Engineer** roles have shown steady, organic growth over the years, reflecting the market's shift toward productionizing AI models.

- **The 2024 Software Engineering Spike:** There is a sudden, massive surge in **Software Engineer** roles in 2024. This could indicate either a broader market stabilization where foundational engineering expanded, or a diversification of the dataset's sourcing for that year.

---

## 📊 Q1.2: The High-Earning Elite
Excluding executive and management-heavy roles, which top 10 job titles command the highest average salaries? To ensure statistical relevance, only roles with at least 100 records are considered.

In [3]:
run_sql("""
        SELECT job_title,
                ROUND(AVG(salary_in_usd), 2) AS avg_salary,
                COUNT(*) AS total_employees
        FROM salaries
        WHERE job_title NOT ILIKE '%manager%' 
                AND job_title NOT ILIKE '%director%'
                AND job_title NOT ILIKE '%head%'
                AND job_title NOT ILIKE '%lead%'
                AND experience_level <> 'EX'
        GROUP BY 1
        HAVING COUNT(*) >= 100
        ORDER BY 2 DESC
        LIMIT 10;
""")

,job_title,avg_salary,total_employees
0,AI Architect,222250.47,102
1,Research Engineer,204332.24,872
2,Research Scientist,197724.57,1958
3,Machine Learning Engineer,196676.16,5025
4,Backend Engineer,194523.67,111
5,Site Reliability Engineer,193740.41,160
6,Solutions Architect,191888.45,264
7,Software Engineer,189988.48,5779
8,Machine Learning Scientist,180871.51,266
9,Applied Scientist,179548.36,1380


💡 Data Insight:

- **The AI & Research Premium:** After filtering for statistical significance (minimum of 100 occurrences), specialized roles in *Machine Learning, AI, and Research* emerge as the highest-paying non-executive positions. When cross-referenced with **Q1.1**, we see these roles have smaller talent pools than general *Data Science*, indicating a high barrier to entry and a premium paid for scarce, highly specialized skill sets.

- **The Infrastructure Backbone (SRE):** Notably, **Site Reliability Engineer** (SRE) ranks 6th. This highlights the market's critical valuation of **infrastructure stability**. As companies scale and deploy complex systems (especially heavy AI/ML models in production) keeping platforms stable, resilient, and continuously online becomes a top-dollar priority.

- **Core Engineering Strength:** Traditional technical roles, specifically **Backend and Software Engineering**, remain highly lucrative anchor positions, proving that foundational product development continues to command premium compensation alongside cutting-edge data fields.

---

## 📊 Q1.3: Emerging Tech Roles
Are there any specialized job titles that had zero presence in 2020/2021 but started appearing frequently in 2023/2024?

In [4]:
run_sql("""
    WITH count_jobs_2020_2021 AS (
        SELECT work_year, job_title, COUNT(*) AS total_employees
        FROM salaries
        WHERE work_year < 2022
        GROUP BY 1, 2
        ), count_jobs_2022_2023_2024 AS (
            SELECT work_year, job_title, COUNT(*) AS total_employees
            FROM salaries
            WHERE work_year >= 2022
            GROUP BY 1, 2
        )

        SELECT COALESCE(c1.job_title, c2.job_title) AS job_title,
                COALESCE(c1.total_employees, 0) AS total_2020_2021,
                COALESCE(c2.total_employees, 0) AS total_2022_to_2024
        FROM count_jobs_2020_2021 AS c1
        FULL JOIN count_jobs_2022_2023_2024 AS c2
            ON c2.job_title = c1.job_title
        WHERE total_2020_2021 = 0
        ORDER BY total_2022_to_2024 DESC
        LIMIT 10
""")

,job_title,total_2020_2021,total_2022_to_2024
0,Software Engineer,0,5817
1,Engineer,0,2482
2,Manager,0,1556
3,Applied Scientist,0,1082
4,Analyst,0,751
5,Research Engineer,0,700
6,Product Manager,0,686
7,Analytics Engineer,0,671
8,Associate,0,622
9,AI Engineer,0,443


💡 Data Insight:

- **Data Sourcing & Classification Shifts:** The appearance of zero records for foundational roles like *Software Engineer*, *Engineer*, and *Analyst* in 2020-2021 points heavily toward a **change in the dataset's sourcing methodologies** or job categorization after 2022, rather than a macroeconomic shift.

- **True Emerging Tech Roles:** Beyond generic titles, the data highly reflects the post-2022 AI and Advanced Analytics boom. *Applied Scientist* (1,082), *Research Engineer* (700), and *AI Engineer* (443) emerged from non-existence to critical volume.

- **The Rise of MLOps & R&D:** The transition of AI from theoretical labs to commercial deployment explains why roles focused on Applied and Research engineering scaled so rapidly during this exact window, matching the massive investments in generative AI and LLM infrastructures.

---

# 🏢 2. Theme 2: Company Size vs. Compensation
## 📊 Q2.1: The Corporate Salary Premium
What is the percentage difference (salary premium) that Large companies (L) pay compared to Small companies/Startups (S) for the same experience level?

In [5]:
run_sql("""
    WITH company_L AS (
        SELECT experience_level,
                ROUND(AVG(salary_in_usd) / 1000, 2) AS salary_L
            FROM salaries
            WHERE company_size = 'L'
            GROUP BY 1
    ), company_S AS (
        SELECT experience_level,
                ROUND(AVG(salary_in_usd) / 1000, 2) AS salary_S
            FROM salaries
            WHERE company_size = 'S'
            GROUP BY 1
    )

        SELECT 
            cl.experience_level,
            salary_L AS salary_L_in_K,
            salary_S AS salary_S_in_K,
            salary_L - salary_S AS diff_in_K,
            100 - (ROUND(salary_S / salary_L * 100, 2)) AS diff_pct,
            RANK() OVER (ORDER BY diff_pct DESC) AS rank_highest_diff_pct
        FROM company_L AS cl
        JOIN company_S AS cs
        ON cs.experience_level = cl.experience_level
        ORDER BY 
            CASE cs.experience_level
            WHEN 'EX' THEN 1
            WHEN 'SE' THEN 2
            WHEN 'MI' THEN 3
            WHEN 'EN' THEN 4
            END ASC
""")

,experience_level,salary_L_in_K,salary_S_in_K,diff_in_K,diff_pct,rank_highest_diff_pct
0,EX,173.80,169.17,4.63,2.66,4
1,SE,170.21,111.19,59.02,34.67,2
2,MI,145.47,74.25,71.22,48.96,1
3,EN,98.62,65.39,33.23,33.69,3


💡 Data Insight:
- **The Overall Corporate Premium:** On average, Large companies (L) pay 30% more than Small companies (S) for equivalent experience levels, validating the financial advantage of corporate compensation packages.

- **The Mid-Level Gulp:** The variance is most severe at the Mid-level (MI), where Small companies pay nearly 49% less than Large enterprises. This suggests that startups might struggle to compete financially for independent, execution-ready talent, making corporate offers significantly more attractive to mid-level professionals.

- **The Executive Compensation Anomaly:** Interestingly, Executive-level (EX) roles pay almost the same regardless of company size (a minimal 2.66% difference). This indicates that early-stage or small companies are willing to pay market-top dollars to secure seasoned leadership to guide their growth.

- **The Senior Salary Ceiling:** In Large companies, Senior professionals (SE) earn an average of USD170.21K, which is remarkably close to what Executives (EX) make (USD173.80K). This implies a high-earning plateau for individual contributors before transitioning into formal executive management.

---

## 📊 Q2.2: Hiring Preferences (Startups vs. Corporates)
Do startups prefer hiring Entry-level (EN) professionals, or do they lean towards Mid/Senior levels? How is the experience level distributed across different company sizes?

In [6]:
run_sql("""
        WITH count_level_per_size AS (
            SELECT company_size,
                    experience_level,
                    COUNT(*) AS total_employees
            FROM salaries
            GROUP BY 1, 2
        )

        SELECT company_size,
                experience_level,
                total_employees,
                ROUND(total_employees / 
                    SUM(total_employees) OVER (PARTITION BY company_size), 2) * 100
                AS pct_employees,
                RANK() OVER (PARTITION BY company_size
                                ORDER BY total_employees DESC
                ) AS rank_pct
        FROM count_level_per_size
        ORDER BY CASE company_size
                    WHEN 'S' THEN 1
                    WHEN 'M' THEN 2
                    WHEN 'L' THEN 3
                    END ASC,
                rank_pct
        
""")

,company_size,experience_level,total_employees,pct_employees,rank_pct
0,S,MI,73,36.0,1
1,S,SE,67,33.0,2
2,S,EN,56,27.0,3
3,S,EX,8,4.0,4
4,M,SE,32846,60.0,1
5,M,MI,16057,29.0,2
6,M,EN,4959,9.0,3
7,M,EX,1163,2.0,4
8,L,SE,952,48.0,1
9,L,MI,770,39.0,2


💡 Data Insight:

- **The Universal Talent Pyramid:** Medium (M) and Large (L) companies follow an identical organizational structure in terms of volume: ```Senior (SE) > Mid-level (MI) > Entry-level (EN) > Executive (EX)```. This reflects a standardized industry blueprint, where corporations maintain a heavy core of experienced executioners supported by a smaller layer of junior talent and lean leadership.

- **Startup Hiring Priorities:** Small companies (S) also lean heavily toward **Senior and Mid-level** professionals to build their foundational products. However, they allocate a higher relative proportion to Entry-level talent compared to Large corporations, likely balancing budget constraints with the need for immediate hands-on builders.

- **Statistical Caveat (Sample Size Disparity):** As a data consultant, it is crucial to note that the total volume of records for *Small companies* in this dataset is **significantly lower** than that of Medium and Large companies. While the percentage trends are highly informative, conclusions regarding startups should consider this sample size limitation.

---

# 🏠 3. Theme 3: The Post-Pandemic Remote Work Paradox
## 📊 Q3.1: Remote vs. In-Person Salary Comparison
Do 100% remote professionals earn less, more, or the same as on-site/hybrid professionals within the same seniority level?

In [7]:
run_sql("""
        SELECT experience_level,
                CASE remote_ratio
                    WHEN 0 THEN 'On-site'
                    WHEN 50 THEN 'Hybrid'
                    WHEN 100 THEN 'Home-office'
                END AS work_model ,
                ROUND(AVG(salary_in_usd) / 1000, 2) AS avg_salary_in_K,
                RANK() OVER (
                    PARTITION BY experience_level
                    ORDER BY avg_salary_in_K DESC) AS rank_avg_salary,
                COUNT(*) AS total_employees,
        FROM salaries
        GROUP BY 1, 2
        ORDER BY 
                CASE experience_level
                WHEN 'EX' THEN 1
                WHEN 'SE' THEN 2
                WHEN 'MI' THEN 3
                WHEN 'EN' THEN 4
                END ASC,
                avg_salary_in_K DESC
""")

,experience_level,work_model,avg_salary_in_K,rank_avg_salary,total_employees
0,EX,Home-office,216.67,1,301
1,EX,On-site,197.96,2,886
2,EX,Hybrid,147.37,3,9
3,SE,On-site,177.95,1,25332
4,SE,Home-office,162.11,2,8462
5,SE,Hybrid,106.40,3,71
6,MI,On-site,148.55,1,13806
7,MI,Home-office,124.34,2,2991
8,MI,Hybrid,77.84,3,103
9,EN,On-site,108.32,1,4331


💡 Data Insight:

- **The "Remote Discount" Pattern:** For Senior (SE), Mid-level (MI), and Entry-level (EN) roles, **On-site** positions consistently command the highest average salaries, followed by **Home-office.** This data supports the market theory of a flexibility premium—where non-executive professionals often accept slightly lower wages in exchange for the geographical freedom and cost savings of working from home.

- **The Executive Privilege:** Executive-level (EX) roles completely invert this trend, with Home-office taking the top spot ($216.67K). This indicates that elite, high-demand leadership talent possesses enough market leverage to command top-tier compensation packages without geographic constraints.

- **The Hybrid Lag:** Across all seniority tiers, Hybrid roles rank **last** both in average salary and sample volume. This suggests that in this dataset, hybrid models might be concentrated in smaller regional markets or organizations with tighter tech budgets, contrasting heavily with the global scale of full-remote or full-office corporate compensation.

---

## 📊 Q3.2: Remote Flexibility by Company Size
Are Medium-sized companies (M) more open to 100% remote work models than Large corporations (L)?

In [8]:
run_sql("""
        WITH total_employees_per_size AS (
            SELECT company_size,
                    remote_ratio,
                    COUNT(*) AS total_employees
            FROM salaries
            WHERE company_size <> 'S'
            GROUP BY 1, 2
        )

        SELECT company_size,
                CASE WHEN remote_ratio = 0 THEN 'On-site'
                    WHEN remote_ratio = 50 THEN 'Hybrid'
                    WHEN remote_ratio = 100 THEN 'Home-office'
                    END AS work_model,
                total_employees,
                ROUND(total_employees /
                    SUM(total_employees) OVER (PARTITION BY company_size)
                    * 100, 2) AS pct_total_employees_per_size,
                RANK() OVER (
                    PARTITION BY company_size
                        ORDER BY total_employees DESC) AS rank_total_employees
        FROM total_employees_per_size
        ORDER BY 1, 5 
        
""")

,company_size,work_model,total_employees,pct_total_employees_per_size,rank_total_employees
0,L,On-site,1474,75.01,1
1,L,Home-office,322,16.39,2
2,L,Hybrid,169,8.60,3
3,M,On-site,42831,77.84,1
4,M,Home-office,12133,22.05,2
5,M,Hybrid,61,0.11,3


💡 Data Insight:

- **The Corporate Mandate:** Both Medium (M) and Large (L) corporations exhibit a very similar cultural baseline, with the **vast majority** of their workforce operating under an **On-site** model (~77.8% for Medium and ~75.0% for Large). This indicates a post-pandemic market stabilization where office presence remains the primary operational standard.

- **Medium Companies Lean Into Full Remote:** Medium-sized companies show a stronger preference for **binary work structures**. When they offer flexibility, they lean almost exclusively into Home-office (22.05%), completely bypassing the logistical overhead of Hybrid models (only 0.11%).

- **Large Corporations and Hybrid Logos:** Large companies show slightly more operational diversity. While their **full-remote** ratio is higher (~16.4%), they still maintain a higher commitment to **Hybrid environments** (8.6%) compared to Medium companies. This likely reflects their capacity to manage the complex real estate and tracking logistics required for hybrid schedules.

---

# 🚀 4. Theme 4: Career Progression ROI
## 📊 Q4.1: The Seniority Salary Leap
What is the average percentage salary increase when a professional progresses from Entry-level (EN) to Mid-level (MI), and from Mid-level to Senior (SE)? Which specific job title yields the highest financial return upon promotion?

In [9]:
run_sql("""
        WITH avg_salary_per_level AS (
            SELECT experience_level,
                    ROUND(AVG(salary_in_usd) / 1000, 2) AS avg_salary_in_K
            FROM salaries
            GROUP BY 1
            ORDER BY CASE experience_level
                        WHEN 'EN' THEN 1
                        WHEN 'MI' THEN 2
                        WHEN 'SE' THEN 3
                        WHEN 'EX' THEN 4
                        END ASC
        )

        SELECT experience_level,
                avg_salary_in_K,
                CASE
                    WHEN experience_level = 'EN' THEN 0
                    ELSE 
                        ROUND(
                            100 -(LAG(avg_salary_in_K) OVER () ) / avg_salary_in_K * 100, 2) 
                    END
                    AS increase_in_salary_pct
        FROM avg_salary_per_level
""")

,experience_level,avg_salary_in_K,increase_in_salary_pct
0,EN,104.45,0.00
1,MI,143.83,27.38
2,SE,173.85,17.27
3,EX,202.29,14.06


💡 Data Insight:

- **The "Autonomy" Premium (EN to MI):** The most significant financial milestone in a tech professional's career occurs when moving from **Entry-level (EN) to Mid-level (MI)**, yielding a massive **27.38%** salary jump. This reflects the high market valuation for autonomy; once a junior professional transitions from requiring constant supervision to driving independent execution, companies are willing to pay a heavy premium.

- **The Diminishing Marginal Returns of Seniority:** As professionals advance further, the percentage increase begins to **decelerate**, dropping to 17.27% when moving to Senior (SE), and 14.06% when reaching the Executive (EX) tier. While the **absolute dollar** increase remains high, the growth rate slows down, indicating a maturation ceiling within individual contributor tracks.

- **Strategic Career Planning:** For tech professionals and HR strategists, this data proves that the fastest path to significant compensation growth is **mastering foundational skills early** to escape the entry-level bracket as quickly as possible.

---

# 🏁 5. Conclusions & Strategic Recommendations

This comprehensive analysis of over **57,000 tech salary records** reveals a dynamic global market driven by specialization, infrastructure stability, and evolving operational models. As a data consultant, the key takeaways from this project can be summarized into three strategic pillars:

## 5.1. The Talent & Compensation Landscape

- **AI & Engineering Premium:** Specialized roles in *Machine Learning*, *AI* *Research*, and *Site Reliability Engineering* (SRE) command the highest salary premiums. This indicates a fierce market competition for professionals who can not only build intelligent models but also maintain the high-availability infrastructure required to run them.

- **The Career Growth Engine:** The path from **Entry-Level (EN) to Mid-Level (MI)** provides the **highest return** on investment (ROI) for professionals, boosting wages by **27.38%**. Organizations must optimize their upskilling pipelines to transition junior talent into autonomous contributors efficiently.

## 5.2. Corporate Architecture vs. Startups
- Large and Medium corporations follow an identical, rigid talent pyramid structure (```Senior > Mid > Entry > Executive```).

- Large companies pay a **30% salary premium** on average compared to startups. This gap peaks at the **Mid-level** tier, where startups pay roughly **49% less**, highlighting the massive financial leverage corporations possess when competing for independent talent.

## 5.3. The Reality of Modern Work Models

- **The Post-Pandemic Office Mandate:** The dataset proves that **On-site work** remains the dominant operational standard for both Medium (77%) and Large (75%) organizations.

- **The Remote Paradox:** Non-executive professionals face a "remote discount," earning **slightly less** in **Home-office** roles compared to **On-site**. However, Executive talent holds the leverage to **invert** this trend, commanding top-tier salaries ($216K+ average) while working **fully remote**.

- **Operational Preferences:** When granting geographic flexibility, Medium-sized companies lean heavily into full Home-office (22%) to **avoid** the administrative overhead of **Hybrid models**, whereas Large corporations are better equipped to sustain both Home-office (16%) and Hybrid environments (8.6%).

---
# 👋 Author's Contact & Thank You Note

Thank you for taking the time to explore this data consulting project!

* **Author:** Adilla Lopes
* **Connect with me on LinkedIn:** [linkedin.com/in/adilla-lopes](https://www.linkedin.com/in/maria-adilla-lopes-da-silva-1900b1210/)
* **Check out my GitHub portfolio:** [github.com/adilla-somnia](https://github.com/adilla-somnia)

*Feel free to upvote this notebook if you found the insights useful, and don't hesitate to reach out for connections or feedback!*

#### 📌 Acknowledgment & Project Origin

This project was inspired by an initial guided dataset from **DataCamp**. While the core dataset and foundational concepts originated there, this notebook was significantly expanded to simulate a real-world data consulting scenario.

All advanced SQL implementations (such as deep window functions, cross-sectional calculations using LAG, and custom data-validation logic) as well as the strategic business recommendations were independently developed to showcase full proficiency in data analytics and business intelligence.